# Chapter 8 Lab — Trend Systems on NIFTY

**Purpose:** repeat Kaufman's main Chapter 8 tests on NIFTY 50 daily data and see which conclusions hold.

Rules for this lab (same protocol as Research 002):
- **No optimisation.** Calculation periods are Kaufman's doubling set 5, 10, 20, 40, 80, 160 — we look at the whole profile, never pick a "best" one.
- Signals use data up to the close of day t. Default fill: **next open**. Section 3 compares fills explicitly.
- Costs: 0.03% per side (index futures). Price index only (no dividends, no roll costs).
- Data: `03_nifty_cycle_tools/data/nifty50_daily_2002_2026.csv` (provenance in that folder).

## Learning objectives

1. Test Kaufman's **fat tail** claim on NIFTY runs and trade results.
2. Compare **trendline-direction** and **price-penetration** signals (Table 8.2).
3. Measure how much **entry timing** matters (Table 8.3).
4. Reproduce the **comparison of major trend systems** across calculation periods.
5. Verify that **MA direction = n-day momentum** (p. 368).
6. Try **bands** (Bollinger) and **two trends** (Golden Cross).
7. Describe when each idea fails.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path('../../03_nifty_cycle_tools/data/nifty50_daily_2002_2026.csv')
df = pd.read_csv(DATA, parse_dates=['Date'], index_col='Date')
O, H, L, C = df['Open'], df['High'], df['Low'], df['Close']
COST = 0.0003
PERIODS = [5, 10, 20, 40, 80, 160]
START = df.index[200]          # every test starts on the same date (Kaufman's rule)
live = df.index >= START
print(len(df), 'days;', START.date(), 'to', df.index[-1].date())

In [ ]:
def run(pos, fill='next_open', cost=COST):
    '''pos_t is decided at the close of day t. Returns daily strategy returns, held position, turnover.'''
    pos = pos.reindex(df.index).fillna(0).where(live, 0)
    if fill == 'next_open':
        before, after = pos.shift(2).fillna(0), pos.shift(1).fillna(0)
        r = (1 + before * (O / C.shift(1) - 1)) * (1 + after * (C / O - 1)) - 1
        held, turnover = after, (after - before).abs()
    else:
        lag = 1 if fill == 'close' else 2           # 'close' = filled at the signal close; 'next_close' = one day later
        held = pos.shift(lag).fillna(0)
        r = held * (C / C.shift(1) - 1)
        turnover = (held - held.shift(1).fillna(0)).abs()
    r = (r.fillna(0) - turnover * cost)[live]
    return r, held[live], turnover[live]


def trades(r, held):
    '''Split daily returns into trades: a trade is a run of the same non-zero position.'''
    tid = (held != held.shift()).cumsum()
    g = pd.DataFrame({'r': r, 'held': held, 'tid': tid})
    g = g[g.held != 0]
    out = g.groupby('tid').agg(ret=('r', lambda x: np.prod(1 + x) - 1), bars=('r', 'size'), side=('held', 'first'))
    return out


def profile(pos, fill='next_open'):
    r, held, t = run(pos, fill)
    tr = trades(r, held)
    yrs = len(r) / 252
    eq = (1 + r).cumprod()
    wins, losses = tr.ret[tr.ret > 0], tr.ret[tr.ret <= 0]
    vol = r.std() * np.sqrt(252)
    return {
        'CAGR %': round(100 * (eq.iloc[-1] ** (1 / yrs) - 1), 1),
        'Sharpe': round(r.mean() * 252 / vol, 2) if vol > 0 else np.nan,
        'MaxDD %': round(100 * (eq / eq.cummax() - 1).min(), 1),
        'Trades/yr': round(len(tr) / yrs, 1),
        'Win %': round(100 * len(wins) / max(len(tr), 1), 1),
        'Profit factor': round(wins.sum() / -losses.sum(), 2) if losses.sum() < 0 else np.nan,
        'Avg win/avg loss': round(wins.mean() / -losses.mean(), 2) if len(losses) and len(wins) else np.nan,
        'Bars in winners': round(tr.bars[tr.ret > 0].mean(), 1),
        'Bars in losers': round(tr.bars[tr.ret <= 0].mean(), 1),
    }

bh = pd.Series(1.0, index=df.index)
pd.DataFrame({'Buy & hold': profile(bh)}).T

## 1. The fat tail (pp. 310–312)

**Kaufman:** in a random series, runs follow the coin-toss pattern (half of all runs have length 1, a quarter length 2, …). Real prices have **fewer short runs and more long runs**, and that is what trend systems profit from.

**Test:** count runs of consecutive up or down closes on NIFTY and compare with the random expectation, then with the same daily returns shuffled into random order.

In [ ]:
def run_lengths(changes):
    s = np.sign(changes[changes != 0])
    ids = (s != s.shift()).cumsum()
    return s.groupby(ids).size()

chg = C.diff()[live].dropna()
real = run_lengths(chg)
rng = np.random.default_rng(8)
shuf = run_lengths(pd.Series(rng.permutation(chg.to_numpy())))

k = np.arange(1, 9)
tab = pd.DataFrame({
    'Random expectation %': 100 * 0.5 ** k,
    'NIFTY %': [100 * (real == i).mean() if i < 8 else 100 * (real >= 8).mean() for i in k],
    'NIFTY shuffled %': [100 * (shuf == i).mean() if i < 8 else 100 * (shuf >= 8).mean() for i in k],
}, index=[str(i) if i < 8 else '8+' for i in k]).round(1)
tab.index.name = 'Run length (days)'
print('Number of runs:', len(real), '| longest run:', real.max(), 'days')
tab

**Read the table:** if NIFTY had a fat tail in *daily direction*, its share of long runs would exceed the random column.
Then look at the *trade* distribution of Kaufman's 40-day MA trendline system — the fat tail that matters for profits.

In [ ]:
ma40 = C.rolling(40).mean()
sig40 = np.sign(ma40.diff()).replace(0, np.nan).ffill().fillna(0)
r40, held40, _ = run(sig40)
tr40 = trades(r40, held40)

fig, ax = plt.subplots(figsize=(11, 4))
ax.hist(100 * tr40.ret, bins=50)
ax.axvline(0, color='k', linewidth=1)
ax.set_xlabel('Trade return %')
ax.set_ylabel('Number of trades')
ax.set_title('40-day MA trendline system on NIFTY: distribution of trade returns (Kaufman Fig. 8.2)')
plt.show()

top = tr40.ret.sort_values(ascending=False)
share_top5 = top.head(5).sum() / tr40.ret[tr40.ret > 0].sum()
print(f"Trades: {len(tr40)}, skew {tr40.ret.skew():.2f}")
print(f"Largest trade {100*top.iloc[0]:.1f}%  vs  median loss {100*tr40.ret[tr40.ret<=0].median():.2f}%")
print(f"Top 5 trades = {100*share_top5:.0f}% of all gross profit")
pd.DataFrame({'NIFTY 40-day MA (Kaufman Table 8.4 profile)': profile(sig40)}).T

## 2. Trendline direction vs price penetration (Table 8.2)

- **Trendline:** long when the MA turns up, short when it turns down.
- **Price cross:** long when the close is above the MA, short when below.

In [ ]:
rows = []
for n in PERIODS[:-1]:
    ma = C.rolling(n).mean()
    trend_sig = np.sign(ma.diff()).replace(0, np.nan).ffill().fillna(0)
    cross_sig = np.sign(C - ma).replace(0, np.nan).ffill().fillna(0)
    a, b = profile(trend_sig), profile(cross_sig)
    rows.append({'Period': n,
                 'Trendline trades/yr': a['Trades/yr'], 'Trendline PF': a['Profit factor'], 'Trendline Sharpe': a['Sharpe'],
                 'Cross trades/yr': b['Trades/yr'], 'Cross PF': b['Profit factor'], 'Cross Sharpe': b['Sharpe'],
                 'Extra trades (cross)': f"{100 * (b['Trades/yr'] / a['Trades/yr'] - 1):.0f}%"})
pd.DataFrame(rows).set_index('Period')

## 3. Entry timing: today's close, next open, next close (Table 8.3)

Same trendline signals, three fills. Kaufman: trending markets prefer **today's close**; noisy equity indices can benefit from **waiting**.

In [ ]:
rows = []
for n in PERIODS[:-1]:
    ma = C.rolling(n).mean()
    s = np.sign(ma.diff()).replace(0, np.nan).ffill().fillna(0)
    row = {'Period': n}
    for f, label in [('close', "Today's close"), ('next_open', 'Next open'), ('next_close', 'Next close')]:
        p = profile(s, f)
        row[f'{label} Sharpe'] = p['Sharpe']
        row[f'{label} PF'] = p['Profit factor']
    rows.append(row)
pd.DataFrame(rows).set_index('Period')

## 4. MA direction = n-day momentum (p. 368)

The n-day MA rises exactly when today's price is above the price that drops out of the window:

`MA_t − MA_{t−1} = (P_t − P_{t−n}) / n`

In [ ]:
for n in PERIODS:
    ma = C.rolling(n).mean()
    d = (ma.diff() - (C - C.shift(n)) / n).abs().max()
    same = (np.sign(ma.diff()) == np.sign(C - C.shift(n)))[C.shift(n).notna()].mean()
    print(f'n = {n:3d}: max difference {d:.2e}, same sign on {100*same:.1f}% of days')

## 5. Comparison of major trend systems (pp. 336–349)

Kaufman's systems, always in the market, no stops. Swing breakout (SWG) is left out — it needs a swing-size parameter in percent, which Kaufman shows is unstable.

In [ ]:
def slope(series, n):
    x = np.arange(n) - (n - 1) / 2
    return series.rolling(n).apply(lambda y: np.dot(x, y) / np.dot(x, x), raw=True)

def hold(long_cond, short_cond):
    s = pd.Series(np.nan, index=df.index)
    s[long_cond] = 1
    s[short_cond] = -1
    return s.ffill().fillna(0)

def systems(n):
    ma = C.rolling(n).mean()
    ex = C.ewm(span=n, adjust=False).mean()
    hh, ll = H.rolling(n).max().shift(1), L.rolling(n).min().shift(1)
    return {
        'M  momentum': hold(C > C.shift(n), C < C.shift(n)),
        'MA moving avg': hold(ma > ma.shift(1), ma < ma.shift(1)),
        'EXP exponential': hold(ex > ex.shift(1), ex < ex.shift(1)),
        'NDB breakout': hold((H > hh) & (C > C.shift(1)), (L < ll) & (C < C.shift(1))),
        'LRS reg. slope': hold(slope(C, n) > 0, slope(C, n) < 0),
    }

res = []
for n in PERIODS:
    for name, s in systems(n).items():
        res.append({'System': name, 'Period': n, **profile(s)})
res = pd.DataFrame(res)
for metric in ['Profit factor', 'Sharpe', 'Win %', 'Trades/yr']:
    print(f'\n{metric}')
    print(res.pivot(index='System', columns='Period', values=metric).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, g in res.groupby('System'):
    axes[0].plot(g['Period'], g['Profit factor'], marker='o', label=name)
    axes[1].plot(g['Trades/yr'], g['Profit factor'], marker='o', linestyle='', label=name)
for ax in axes:
    ax.axhline(1, color='k', linewidth=1)
axes[0].set_xscale('log'); axes[0].set_xticks(PERIODS); axes[0].set_xticklabels(PERIODS)
axes[0].set_xlabel('Calculation period'); axes[0].set_ylabel('Profit factor')
axes[0].set_title('Profit factor by period (Kaufman Fig. 8.16–8.19)')
axes[1].set_xlabel('Trades per year'); axes[1].set_title('Fewer trades, better profit factor? (Fig. 8.21)')
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 6. Bands: Bollinger 20 / 2σ (pp. 323–329)

- **Trend use (Kaufman rule 2):** buy on a close above the upper band, exit on a close below the 20-day MA; mirror for shorts.
- **Bollinger's own use (mean reversion):** buy on a close below the lower band, exit at the MA; mirror for shorts.

In [ ]:
mid = C.rolling(20).mean()
sd = C.rolling(20).std()
up, lo = mid + 2 * sd, mid - 2 * sd

def band_system(enter_long, exit_long, enter_short, exit_short):
    pos, out = 0, []
    for el, xl, es, xs in zip(enter_long, exit_long, enter_short, exit_short):
        if pos == 1 and xl: pos = 0
        elif pos == -1 and xs: pos = 0
        if pos == 0:
            pos = 1 if el else (-1 if es else 0)
        out.append(pos)
    return pd.Series(out, index=df.index, dtype=float)

trend_band = band_system(C > up, C < mid, C < lo, C > mid)
mr_band = band_system(C < lo, C > mid, C > up, C < mid)
inside = ((C <= up) & (C >= lo))[live & up.notna()].mean()
print(f'Share of closes inside the 2σ band: {100*inside:.1f}%  (Kaufman: ~87%; normal: 95.4%)')
pd.DataFrame({'Bollinger trend (enter band, exit MA)': profile(trend_band),
              'Bollinger trend long-only': profile(trend_band.clip(lower=0)),
              'Bollinger mean reversion': profile(mr_band),
              'Bollinger mean reversion long-only': profile(mr_band.clip(lower=0)),
              'Buy & hold': profile(bh)}).T

## 7. Two trends: the Golden Cross (p. 354)

Long when the 50-day MA is above the 200-day, flat otherwise (Kaufman earns T-bills when flat; we earn nothing, which understates it).

In [ ]:
gc = (C.rolling(50).mean() > C.rolling(200).mean()).astype(float)
ma40_long = sig40.clip(lower=0)
summary = pd.DataFrame({'Golden Cross 50/200 long-only': profile(gc),
                        '40-day MA trendline long-only': profile(ma40_long),
                        '40-day MA trendline L/S': profile(sig40),
                        'Buy & hold': profile(bh)}).T
summary

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
for name, s in [('Buy & hold', bh), ('Golden Cross long-only', gc), ('40-day MA long-only', ma40_long), ('40-day MA L/S', sig40)]:
    r, _, _ = run(s)
    ax.plot((1 + r).cumprod(), label=name)
ax.set_yscale('log'); ax.set_ylabel('Growth of 1 (log)'); ax.legend()
ax.set_title('NIFTY: trend systems vs buy & hold (next-open fills, costs)')
plt.show()

In [ ]:
# sanity checks
assert all((C.rolling(n).mean().diff() - (C - C.shift(n)) / n).abs().max() < 1e-6 for n in PERIODS)
assert set(sig40.unique()) <= {-1.0, 0.0, 1.0}
assert res['Trades/yr'].groupby(res['Period']).mean().is_monotonic_decreasing
print('Chapter 8 sanity checks passed.')

## 8. What the NIFTY results say (run 2026-10-20, 2002-10 to 2026-03)

| Kaufman's claim | NIFTY result | Holds? |
|---|---|---|
| Prices have more long runs than random (fat tail) | Runs of 6+ days: **5.1%** of runs vs 2.8% random and 2.7% shuffled | ✓ |
| Trend profits come from a few large trades | 40-day MA: skew 5.8, largest trade +81%, **top 5 trades = 31%** of gross profit | ✓ |
| Typical profile: ~35% winners, wins ≫ losses, winners held longer | 37.6% winners, win/loss 2.79, winners **35.6** bars vs losers **8.3** | ✓ |
| Trendline beats price cross (fewer trades, better results) | Cross had 13–37% more trades but a **higher Sharpe at every period** | ✗ on NIFTY |
| Noisy equity index: delaying entry can help | **Delay hurt**: today's close best (Sharpe 0.62 at 20 days), next open 0.44, next close 0.16 | ✗ on NIFTY |
| MA direction = n-day momentum | Identical on 100% of days; identical results | ✓ (exact) |
| Longer periods → better profit factor | Rises from 5 to 80 days for all systems; falls at 160 except breakout | ✓ up to 80 |
| Breakout: most reliable, fewest trades | Highest win % (~40%), fewest trades (0.7–25/yr) | ✓ |
| Bollinger 2σ holds ~87% | **90.8%** of closes inside | ≈ |
| Bands as trend tool vs mean reversion | Trend use profitable; mean reversion lost 7.6%/yr despite 59.5% winners (avg win/loss 0.41) | trend ✓ |

**Practical points:**
- No long/short trend system beat buy & hold's Sharpe (0.76): shorting a market that rose ~14% a year is expensive.
- Long-only trend filters reduced drawdown sharply: 40-day MA long-only Sharpe **0.96**, max drawdown −36% vs −59%.
- The Golden Cross was *worse* than buy & hold here (0.61) — but it earns nothing when flat in this test; Kaufman credits T-bills.
- NDB at 160 days shows profit factor 3.49 from about 16 trades in 23 years — too few to trust.
- Momentum (green) is hidden under the MA line in the chart because the two systems are identical.